# AAT6153 Chapter 1 — Hands-on 4
## LeNet: a "Hello World" Convolutional Neural Network
**Slides:** Typical CNN Architecture (slides 47–52), LeNet (slides 53–54), Saving and Loading CNN Models (slides 55–57)

In this notebook you will:
1. Build **LeNet** in Keras and read its layer pattern `INPUT → [CONV → ACT → POOL] × 2 → FC → ACT → FC → ACT → FC`.
2. Train it on MNIST with **early stopping** and evaluate it with a **classification report** and **confusion matrix**.
3. **Visualise** the kernels LeNet has learned and the feature maps they produce.
4. **Save and load** the model, both as a single `.keras` file and as JSON + HDF5 weights.

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** recommended.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

tf.keras.utils.set_random_seed(7)

## Load and preprocess the data

In [ ]:
(training_images, training_labels), (test_images, test_labels) = mnist.load_data()

# Display a handwritten digit in the training set (change m to display another digit)
m = 111
plt.axis(False)
plt.title(f'Label: {training_labels[m]}')
plt.imshow(training_images[m], cmap='gray')
plt.show()

In [ ]:
# CNNs expect (height, width, channels): add a channel axis and scale pixels to 0–1.
# Unlike the MLP in Hands-on 2, we do NOT flatten the image. The 2-D structure is kept.
training_images = training_images.reshape(-1, 28, 28, 1) / 255.0
test_images = test_images.reshape(-1, 28, 28, 1) / 255.0
print(training_images.shape, test_images.shape)

## Define LeNet (slide 54)
The original LeNet used tanh and average pooling; this version uses ReLU and max pooling.

In [ ]:
model = models.Sequential([
    # Input layer
    layers.Input(shape=(28, 28, 1)),
    # C1: Convolutional layer
    layers.Conv2D(filters=6, kernel_size=(5, 5), activation='relu', padding='same'),
    # S2: Pooling (original: average pooling)
    layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    # C3: Convolutional layer
    layers.Conv2D(filters=16, kernel_size=(5, 5), activation='relu'),
    # S4: Pooling (original: average pooling)
    layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    # Flatten before entering fully connected layers
    layers.Flatten(),
    # C5: Fully connected layer (a 5x5 conv layer in the original)
    layers.Dense(120, activation='relu'),
    # F6: Fully connected layer
    layers.Dense(84, activation='relu'),
    # Output layer
    layers.Dense(10, activation='softmax')
])
model.summary()

**Check your understanding**
* Why is C1's output 28 × 28 but C3's output 10 × 10? (Hint: `padding='same'` vs the default `'valid'`, and the output-size formula from Hands-on 3.)
* C1 has 6 kernels of 5 × 5 × 1 plus 6 biases = **156** parameters. Compare LeNet's total with the 615k+ parameters of the MLP in Hands-on 2.

## Compile and train
We use **integer labels** with `sparse_categorical_crossentropy`, so no one-hot encoding is needed.

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Use EarlyStopping to prevent overfitting (slide 15)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(training_images, training_labels, validation_split=0.1,
                    epochs=20, batch_size=64, callbacks=[early_stop])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], 'b', label='train')
ax1.plot(history.history['val_accuracy'], 'g', label='val')
ax1.set(xlabel='Epoch', ylabel='Accuracy', title='Model Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], 'b', label='train')
ax2.plot(history.history['val_loss'], 'g', label='val')
ax2.set(xlabel='Epoch', ylabel='Loss', title='Model Loss'); ax2.legend()
plt.show()

## Evaluate on the test set

In [ ]:
score = model.evaluate(test_images, test_labels, verbose=0)
print(f'Loss: {score[0]:.4f}, accuracy: {score[1]:.3%}')

# Convert probabilities to predicted classes
pred_labels = np.argmax(model.predict(test_images, verbose=0), axis=1)

print("\nClassification Report:")
print(classification_report(test_labels, pred_labels, digits=4))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(test_labels, pred_labels, cmap='Blues', ax=ax)
ax.set_title("Confusion Matrix for LeNet on MNIST")
plt.show()

In [ ]:
# Show some of the digits LeNet got wrong
wrong = np.where(pred_labels != test_labels)[0]
print(f"{len(wrong)} misclassified test images out of {len(test_labels)}")
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, i in zip(axes, wrong[:8]):
    ax.imshow(test_images[i].squeeze(), cmap='gray'); ax.axis(False)
    ax.set_title(f"true {test_labels[i]} / pred {pred_labels[i]}", fontsize=9)
plt.show()

---
## What has the CNN learned? (slides 43–44)
### Learned kernels of the first convolutional layer
Unlike the hand-designed Sobel/Laplacian kernels in Hands-on 3, these six 5 × 5 kernels were **learned** from data.

In [ ]:
kernels, biases = model.layers[0].get_weights()        # shape (5, 5, 1, 6)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for i, ax in enumerate(axes):
    ax.imshow(kernels[:, :, 0, i], cmap='gray'); ax.axis(False)
    ax.set_title(f"C1 kernel {i}")
plt.show()

### Feature maps
Feed one digit through the network and look at the output of each conv layer. C1 responds to simple strokes and edges; C3 combines them into larger patterns.

In [ ]:
feature_model = models.Model(inputs=model.inputs,
                             outputs=[model.layers[0].output, model.layers[2].output])
digit = test_images[0:1]
c1_maps, c3_maps = feature_model.predict(digit, verbose=0)

plt.imshow(digit.squeeze(), cmap='gray'); plt.title(f"Input (label {test_labels[0]})"); plt.axis(False); plt.show()

fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for i, ax in enumerate(axes):
    ax.imshow(c1_maps[0, :, :, i], cmap='viridis'); ax.axis(False); ax.set_title(f"C1 map {i}")
plt.suptitle("C1 feature maps (28×28)"); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(c3_maps[0, :, :, i], cmap='viridis'); ax.axis(False); ax.set_title(f"C3 map {i}", fontsize=9)
plt.suptitle("C3 feature maps (10×10)"); plt.show()

---
## Saving and loading the model (slides 55–57)
### Option 1: the whole model in one `.keras` file (recommended)

In [ ]:
model.save("lenet_mnist.keras")
reloaded = tf.keras.models.load_model("lenet_mnist.keras")
print("Reloaded accuracy:", reloaded.evaluate(test_images, test_labels, verbose=0)[1])

### Option 2: architecture as JSON + weights as HDF5 (slide 57)

In [ ]:
# serialize model to JSON
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("model.weights.h5")
print("Saved model to disk")

In [ ]:
from tensorflow.keras.models import model_from_json

# load json and create model
with open('model.json', 'r') as json_file:
    loaded_model = model_from_json(json_file.read())
# load weights into new model
loaded_model.load_weights("model.weights.h5")
print("Loaded model from disk")

# A model rebuilt from JSON must be compiled again before evaluate()
loaded_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Loaded-model accuracy:", loaded_model.evaluate(test_images, test_labels, verbose=0)[1])

### Predict a single test image
Change `index` to any value between 0 and 9999.

In [ ]:
index = 4321

x = np.expand_dims(test_images[index], axis=0)
probs = loaded_model.predict(x, verbose=0)
print(f'Predicted label: {probs.argmax()}  (confidence {probs.max():.2%})')

plt.axis(False)
plt.title(f'Actual label: {test_labels[index]}')
plt.imshow(test_images[index].squeeze(), cmap='gray')
plt.show()

---
## Try it yourself: Batch Normalization and Dropout (slides 47, 50–52)
Build a modified LeNet that follows the pattern

`INPUT → [CONV → ACT → BN → POOL] × 2 → FLATTEN → [FC → ACT → DO] × 2 → FC`

Complete the **To do** lines, train it with the same settings, and compare its accuracy and number of epochs with the original LeNet.

In [ ]:
model_bn = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, (5, 5), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    # To do: add the second CONV → BN → POOL block (16 filters, 5x5)

    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dropout(0.3),
    # To do: add Dense(84, relu) followed by Dropout(0.3)

    layers.Dense(10, activation='softmax'),
])
model_bn.summary()

# To do: compile, train with EarlyStopping, and evaluate on the test set